In [5]:
# ============================================================
# J12 — MLflow Tracking — OCP Slurry Pipeline
# CELLULE 1/12 — Imports
# ============================================================

import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    f1_score, roc_auc_score,
    precision_score, recall_score,
    confusion_matrix, ConfusionMatrixDisplay
)
import joblib
import os

print("="*45)
print("  CELLULE 1 — Imports")
print("="*45)
print(f"  MLflow  : {mlflow.__version__}")
print(f"  Pandas  : {pd.__version__}")
print(f"  Numpy   : {np.__version__}")
print("="*45)
print("  ✅ Cellule 1 OK — passe à la cellule 2")
print("="*45)

  CELLULE 1 — Imports
  MLflow  : 3.12.0
  Pandas  : 2.3.3
  Numpy   : 2.4.4
  ✅ Cellule 1 OK — passe à la cellule 2


In [6]:
# ============================================================
# CELLULE 2/12 — Chargement données + features physiques
# ============================================================

# Chargement dataset
df = pd.read_csv("data/ai4i2020.csv")

# Features physiques dérivées — identiques à J11
df["temp_diff"]          = df["Process temperature [K]"] - df["Air temperature [K]"]
df["power_estimated_w"]  = df["Torque [Nm]"] * df["Rotational speed [rpm]"] * (2 * np.pi / 60)
df["tool_wear_rate"]     = df["Tool wear [min]"] / (df["Rotational speed [rpm]"] + 1)
df["torque_speed_ratio"] = df["Torque [Nm]"] / (df["Rotational speed [rpm]"] + 1)

# Encodage Type L/M/H
df = pd.get_dummies(df, columns=["Type"], prefix="station")

# Features sélectionnées
FEATURES = [
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]",
    "temp_diff",
    "power_estimated_w",
    "tool_wear_rate",
    "torque_speed_ratio",
    "station_H",
    "station_L",
    "station_M"
]

TARGET = "Machine failure"

features_ok  = [f for f in FEATURES if f in df.columns]
features_nok = [f for f in FEATURES if f not in df.columns]

X = df[FEATURES]
y = df[TARGET]

print("="*45)
print("  CELLULE 2/12 — Données")
print("="*45)
print(f"  Lignes           : {df.shape[0]}")
print(f"  Colonnes totales : {df.shape[1]}")
print(f"  Features OK      : {len(features_ok)}/12")
if features_nok:
    print(f"  ⚠️  Manquantes  : {features_nok}")
else:
    print(f"  Features KO      : 0 ✅")
print(f"  Pannes           : {y.sum()} / {len(y)} ({y.mean()*100:.1f}%)")
print("="*45)
print("  ✅ Cellule 2 OK — passe à la cellule 3")
print("="*45)

  CELLULE 2/12 — Données
  Lignes           : 10000
  Colonnes totales : 20
  Features OK      : 12/12
  Features KO      : 0 ✅
  Pannes           : 339 / 10000 (3.4%)
  ✅ Cellule 2 OK — passe à la cellule 3


In [7]:
# ============================================================
# CELLULE 3/12 — Split stratifié + StandardScaler
# random_state=42 obligatoire pour reproductibilité
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size    = 0.2,
    random_state = 42,
    stratify     = y
)

scaler         = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print("="*45)
print("  CELLULE 3/12 — Split + Scaler")
print("="*45)
print(f"  Train            : {X_train.shape[0]} lignes")
print(f"  Test             : {X_test.shape[0]} lignes")
print(f"  Pannes train     : {y_train.sum()} ({y_train.mean()*100:.1f}%)")
print(f"  Pannes test      : {y_test.sum()} ({y_test.mean()*100:.1f}%)")
print(f"  Scaler fitted    : ✅")
print("="*45)
print("  ✅ Cellule 3 OK — passe à la cellule 4")
print("="*45)

  CELLULE 3/12 — Split + Scaler
  Train            : 8000 lignes
  Test             : 2000 lignes
  Pannes train     : 271 (3.4%)
  Pannes test      : 68 (3.4%)
  Scaler fitted    : ✅
  ✅ Cellule 3 OK — passe à la cellule 4


In [8]:
# ============================================================
# CELLULE 4/12 — Fonction helper evaluate_model
# Réutilisée pour les 3 runs — évite la répétition de code
# ============================================================

def evaluate_model(model, X_test_scaled, y_test, threshold=0.5):
    """
    Évalue un modèle avec seuil personnalisé.

    Paramètres :
        model          : modèle sklearn entraîné
        X_test_scaled  : features test normalisées
        y_test         : vraies étiquettes
        threshold      : seuil de décision (défaut 0.5)

    Retourne :
        metrics (dict), y_pred (array), y_proba (array)
    """
    # Probabilité d'appartenir à la classe 1 (panne)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]

    # Application du seuil personnalisé
    y_pred = (y_proba >= threshold).astype(int)

    metrics = {
        "f1_score"  : round(float(f1_score(y_test, y_pred)), 4),
        "auc"       : round(float(roc_auc_score(y_test, y_proba)), 4),
        "precision" : round(float(precision_score(y_test, y_pred, zero_division=0)), 4),
        "recall"    : round(float(recall_score(y_test, y_pred)), 4),
        "threshold" : float(threshold)
    }
    return metrics, y_pred, y_proba

print("="*45)
print("  CELLULE 4/12 — Fonction evaluate_model")
print("="*45)
print("  Métriques calculées :")
print("    → f1_score")
print("    → auc")
print("    → precision")
print("    → recall")
print("    → threshold")
print("="*45)
print("  ✅ Cellule 4 OK — passe à la cellule 5")
print("="*45)

  CELLULE 4/12 — Fonction evaluate_model
  Métriques calculées :
    → f1_score
    → auc
    → precision
    → recall
    → threshold
  ✅ Cellule 4 OK — passe à la cellule 5


In [9]:
# ============================================================
# CELLULE 5/12 — Configuration MLflow
# Tout sera stocké dans ./mlruns (créé automatiquement)
# ============================================================

EXPERIMENT_NAME = "OCP_Maintenance_Predictive"

mlflow.set_tracking_uri("./mlruns")
mlflow.set_experiment(EXPERIMENT_NAME)

print("="*45)
print("  CELLULE 5/12 — Configuration MLflow")
print("="*45)
print(f"  Expérience : {EXPERIMENT_NAME}")
print(f"  Stockage   : ./mlruns")
print("="*45)
print("  ✅ Cellule 5 OK — passe à la cellule 6")
print("="*45)

2026/05/13 07:08:11 INFO mlflow.tracking.fluent: Experiment with name 'OCP_Maintenance_Predictive' does not exist. Creating a new experiment.


  CELLULE 5/12 — Configuration MLflow
  Expérience : OCP_Maintenance_Predictive
  Stockage   : ./mlruns
  ✅ Cellule 5 OK — passe à la cellule 6


In [10]:
# ============================================================
# CELLULE 6/12 — Run 1 : RF Baseline
# Exactement le modèle J11 — sert de référence
# ============================================================

print("⏳ Run 1 : RF Baseline en cours...")

with mlflow.start_run(run_name="RF_Baseline"):

    # Paramètres
    p1 = {
        "n_estimators" : 300,
        "max_depth"    : None,
        "class_weight" : {0: 1, 1: 28},
        "threshold"    : 0.39,
        "random_state" : 42
    }

    # Entraînement
    model1 = RandomForestClassifier(
        n_estimators = p1["n_estimators"],
        max_depth    = p1["max_depth"],
        class_weight = p1["class_weight"],
        random_state = p1["random_state"],
        n_jobs       = -1
    )
    model1.fit(X_train_scaled, y_train)

    # Évaluation
    metrics1, y_pred1, y_proba1 = evaluate_model(
        model1, X_test_scaled, y_test,
        threshold=p1["threshold"]
    )

    # Log MLflow
    mlflow.log_params({
        "n_estimators" : p1["n_estimators"],
        "max_depth"    : "None",
        "class_weight" : "1:28",
        "threshold"    : p1["threshold"]
    })
    mlflow.log_metrics(metrics1)
    mlflow.sklearn.log_model(model1, "model")

    run1_id = mlflow.active_run().info.run_id

print("="*45)
print("  CELLULE 6/12 — Run 1 : RF Baseline")
print("="*45)
print(f"  F1        : {metrics1['f1_score']}")
print(f"  AUC       : {metrics1['auc']}")
print(f"  Precision : {metrics1['precision']}")
print(f"  Recall    : {metrics1['recall']}")
print(f"  Run ID    : {run1_id[:8]}...")
print("="*45)
print("  ✅ Cellule 6 OK — passe à la cellule 7")
print("="*45)

⏳ Run 1 : RF Baseline en cours...


2026/05/13 07:08:14 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/13 07:08:14 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


  CELLULE 6/12 — Run 1 : RF Baseline
  F1        : 0.864
  AUC       : 0.9732
  Precision : 0.9474
  Recall    : 0.7941
  Run ID    : 185a0d61...
  ✅ Cellule 6 OK — passe à la cellule 7


In [11]:
# ============================================================
# CELLULE 6/12 — Run 1 : RF Baseline
# Exactement le modèle J11 — sert de référence
# ============================================================

print("⏳ Run 1 : RF Baseline en cours...")

with mlflow.start_run(run_name="RF_Baseline"):

    # Paramètres
    p1 = {
        "n_estimators" : 300,
        "max_depth"    : None,
        "class_weight" : {0: 1, 1: 28},
        "threshold"    : 0.39,
        "random_state" : 42
    }

    # Entraînement
    model1 = RandomForestClassifier(
        n_estimators = p1["n_estimators"],
        max_depth    = p1["max_depth"],
        class_weight = p1["class_weight"],
        random_state = p1["random_state"],
        n_jobs       = -1
    )
    model1.fit(X_train_scaled, y_train)

    # Évaluation
    metrics1, y_pred1, y_proba1 = evaluate_model(
        model1, X_test_scaled, y_test,
        threshold=p1["threshold"]
    )

    # Log MLflow
    mlflow.log_params({
        "n_estimators" : p1["n_estimators"],
        "max_depth"    : "None",
        "class_weight" : "1:28",
        "threshold"    : p1["threshold"]
    })
    mlflow.log_metrics(metrics1)
    mlflow.sklearn.log_model(model1, "model")

    run1_id = mlflow.active_run().info.run_id

print("="*45)
print("  CELLULE 6/12 — Run 1 : RF Baseline")
print("="*45)
print(f"  F1        : {metrics1['f1_score']}")
print(f"  AUC       : {metrics1['auc']}")
print(f"  Precision : {metrics1['precision']}")
print(f"  Recall    : {metrics1['recall']}")
print(f"  Run ID    : {run1_id[:8]}...")
print("="*45)
print("  ✅ Cellule 6 OK — passe à la cellule 7")
print("="*45)

⏳ Run 1 : RF Baseline en cours...


2026/05/13 07:08:37 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/13 07:08:37 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


  CELLULE 6/12 — Run 1 : RF Baseline
  F1        : 0.864
  AUC       : 0.9732
  Precision : 0.9474
  Recall    : 0.7941
  Run ID    : 06c1915e...
  ✅ Cellule 6 OK — passe à la cellule 7


In [12]:
# ============================================================
# CELLULE 6/12 — Run 1 : RF Baseline
# Exactement le modèle J11 — sert de référence
# ============================================================

print("⏳ Run 1 : RF Baseline en cours...")

with mlflow.start_run(run_name="RF_Baseline"):

    # Paramètres
    p1 = {
        "n_estimators" : 300,
        "max_depth"    : None,
        "class_weight" : {0: 1, 1: 28},
        "threshold"    : 0.39,
        "random_state" : 42
    }

    # Entraînement
    model1 = RandomForestClassifier(
        n_estimators = p1["n_estimators"],
        max_depth    = p1["max_depth"],
        class_weight = p1["class_weight"],
        random_state = p1["random_state"],
        n_jobs       = -1
    )
    model1.fit(X_train_scaled, y_train)

    # Évaluation
    metrics1, y_pred1, y_proba1 = evaluate_model(
        model1, X_test_scaled, y_test,
        threshold=p1["threshold"]
    )

    # Log MLflow
    mlflow.log_params({
        "n_estimators" : p1["n_estimators"],
        "max_depth"    : "None",
        "class_weight" : "1:28",
        "threshold"    : p1["threshold"]
    })
    mlflow.log_metrics(metrics1)
    mlflow.sklearn.log_model(model1, "model")

    run1_id = mlflow.active_run().info.run_id

print("="*45)
print("  CELLULE 6/12 — Run 1 : RF Baseline")
print("="*45)
print(f"  F1        : {metrics1['f1_score']}")
print(f"  AUC       : {metrics1['auc']}")
print(f"  Precision : {metrics1['precision']}")
print(f"  Recall    : {metrics1['recall']}")
print(f"  Run ID    : {run1_id[:8]}...")
print("="*45)
print("  ✅ Cellule 6 OK — passe à la cellule 7")
print("="*45)

⏳ Run 1 : RF Baseline en cours...


2026/05/13 07:08:49 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/13 07:08:49 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


  CELLULE 6/12 — Run 1 : RF Baseline
  F1        : 0.864
  AUC       : 0.9732
  Precision : 0.9474
  Recall    : 0.7941
  Run ID    : cef43bec...
  ✅ Cellule 6 OK — passe à la cellule 7


In [13]:
# ============================================================
# CELLULE 7/12 — Run 2 : RF Optimisé
# max_depth=15 → réduit overfitting
# n_estimators=500 → plus stable
# ============================================================

print("⏳ Run 2 : RF Optimisé en cours...")

with mlflow.start_run(run_name="RF_Optimise"):

    p2 = {
        "n_estimators" : 500,
        "max_depth"    : 15,
        "class_weight" : {0: 1, 1: 28},
        "threshold"    : 0.39,
        "random_state" : 42
    }

    model2 = RandomForestClassifier(
        n_estimators = p2["n_estimators"],
        max_depth    = p2["max_depth"],
        class_weight = p2["class_weight"],
        random_state = p2["random_state"],
        n_jobs       = -1
    )
    model2.fit(X_train_scaled, y_train)

    metrics2, y_pred2, y_proba2 = evaluate_model(
        model2, X_test_scaled, y_test,
        threshold=p2["threshold"]
    )

    mlflow.log_params({
        "n_estimators" : p2["n_estimators"],
        "max_depth"    : p2["max_depth"],
        "class_weight" : "1:28",
        "threshold"    : p2["threshold"]
    })
    mlflow.log_metrics(metrics2)
    mlflow.sklearn.log_model(model2, "model")

    run2_id = mlflow.active_run().info.run_id

print("="*45)
print("  CELLULE 7/12 — Run 2 : RF Optimisé")
print("="*45)
print(f"  F1        : {metrics2['f1_score']}")
print(f"  AUC       : {metrics2['auc']}")
print(f"  Precision : {metrics2['precision']}")
print(f"  Recall    : {metrics2['recall']}")
print(f"  Run ID    : {run2_id[:8]}...")
print("="*45)
print("  ✅ Cellule 7 OK — passe à la cellule 8")
print("="*45)

⏳ Run 2 : RF Optimisé en cours...


2026/05/13 07:09:09 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/13 07:09:09 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


  CELLULE 7/12 — Run 2 : RF Optimisé
  F1        : 0.8548
  AUC       : 0.979
  Precision : 0.9464
  Recall    : 0.7794
  Run ID    : 549c91ff...
  ✅ Cellule 7 OK — passe à la cellule 8


In [14]:
# ============================================================
# CELLULE 8/12 — Run 3 : RF Agressif
# Logique métier OCP : panne non détectée = arrêt pipeline
# On maximise le Recall au détriment de la Precision
# class_weight 1:50 + seuil 0.35 = très sensible aux pannes
# ============================================================

print("⏳ Run 3 : RF Agressif en cours...")

with mlflow.start_run(run_name="RF_Agressif"):

    p3 = {
        "n_estimators" : 300,
        "max_depth"    : None,
        "class_weight" : {0: 1, 1: 50},
        "threshold"    : 0.35,
        "random_state" : 42
    }

    model3 = RandomForestClassifier(
        n_estimators = p3["n_estimators"],
        max_depth    = p3["max_depth"],
        class_weight = p3["class_weight"],
        random_state = p3["random_state"],
        n_jobs       = -1
    )
    model3.fit(X_train_scaled, y_train)

    metrics3, y_pred3, y_proba3 = evaluate_model(
        model3, X_test_scaled, y_test,
        threshold=p3["threshold"]
    )

    mlflow.log_params({
        "n_estimators" : p3["n_estimators"],
        "max_depth"    : "None",
        "class_weight" : "1:50",
        "threshold"    : p3["threshold"]
    })
    mlflow.log_metrics(metrics3)
    mlflow.sklearn.log_model(model3, "model")

    run3_id = mlflow.active_run().info.run_id

print("="*45)
print("  CELLULE 8/12 — Run 3 : RF Agressif")
print("="*45)
print(f"  F1        : {metrics3['f1_score']}")
print(f"  AUC       : {metrics3['auc']}")
print(f"  Precision : {metrics3['precision']}")
print(f"  Recall    : {metrics3['recall']}")
print(f"  Run ID    : {run3_id[:8]}...")
print("="*45)
print("  ✅ Cellule 8 OK — passe à la cellule 9")
print("="*45)

⏳ Run 3 : RF Agressif en cours...


2026/05/13 07:09:22 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/13 07:09:22 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


  CELLULE 8/12 — Run 3 : RF Agressif
  F1        : 0.8346
  AUC       : 0.9719
  Precision : 0.8983
  Recall    : 0.7794
  Run ID    : 64e0b663...
  ✅ Cellule 8 OK — passe à la cellule 9


In [15]:
# ============================================================
# CELLULE 9/12 — Tableau comparatif des 3 runs
# C'est ce qu'on montre au manager et dans le portfolio
# ============================================================

results = pd.DataFrame({
    "Run"       : ["RF_Baseline", "RF_Optimise", "RF_Agressif"],
    "F1"        : [metrics1["f1_score"],  metrics2["f1_score"],  metrics3["f1_score"]],
    "AUC"       : [metrics1["auc"],       metrics2["auc"],       metrics3["auc"]],
    "Precision" : [metrics1["precision"], metrics2["precision"], metrics3["precision"]],
    "Recall"    : [metrics1["recall"],    metrics2["recall"],    metrics3["recall"]],
    "Threshold" : [metrics1["threshold"], metrics2["threshold"], metrics3["threshold"]]
})

# Meilleur modèle selon F1
best_idx   = results["F1"].idxmax()
best_run   = results.loc[best_idx, "Run"]
best_f1    = results.loc[best_idx, "F1"]
best_auc   = results.loc[best_idx, "AUC"]
best_model = [model1, model2, model3][best_idx]

print("="*65)
print("        CELLULE 9/12 — Comparaison 3 Expériences MLflow")
print("="*65)
print(results.to_string(index=False))
print("="*65)
print(f"\n  🏆 Meilleur modèle : {best_run}")
print(f"     F1  = {best_f1}")
print(f"     AUC = {best_auc}")
print("="*65)
print("  ✅ Cellule 9 OK — passe à la cellule 10")
print("="*65)

        CELLULE 9/12 — Comparaison 3 Expériences MLflow
        Run     F1    AUC  Precision  Recall  Threshold
RF_Baseline 0.8640 0.9732     0.9474  0.7941       0.39
RF_Optimise 0.8548 0.9790     0.9464  0.7794       0.39
RF_Agressif 0.8346 0.9719     0.8983  0.7794       0.35

  🏆 Meilleur modèle : RF_Baseline
     F1  = 0.864
     AUC = 0.9732
  ✅ Cellule 9 OK — passe à la cellule 10


In [16]:
# ============================================================
# CELLULE 10/12 — Visualisation des métriques
# Pour portfolio LinkedIn et présentation manager
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle(
    "OCP Maintenance Prédictive — Comparaison MLflow",
    fontsize=14, fontweight='bold'
)

colors = ['#2196F3', '#4CAF50', '#FF9800']
runs   = results["Run"]
x      = np.arange(len(runs))
width  = 0.35

# Graphique 1 — F1 vs AUC
axes[0].bar(x - width/2, results["F1"],  width,
            label="F1-Score", color=colors, alpha=0.85)
axes[0].bar(x + width/2, results["AUC"], width,
            label="AUC",      color=colors, alpha=0.5)
axes[0].set_xticks(x)
axes[0].set_xticklabels(runs, rotation=10)
axes[0].set_ylim(0.7, 1.05)
axes[0].set_title("F1-Score vs AUC")
axes[0].legend()
axes[0].axhline(y=0.85, color='red', linestyle='--',
                alpha=0.5, label="Objectif F1=0.85")
axes[0].grid(axis='y', alpha=0.3)

# Graphique 2 — Precision vs Recall
axes[1].bar(x - width/2, results["Precision"], width,
            label="Precision", color=colors, alpha=0.85)
axes[1].bar(x + width/2, results["Recall"],    width,
            label="Recall",    color=colors, alpha=0.5)
axes[1].set_xticks(x)
axes[1].set_xticklabels(runs, rotation=10)
axes[1].set_ylim(0, 1.15)
axes[1].set_title("Precision vs Recall")
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig("mlflow_comparison.png", dpi=150, bbox_inches='tight')
plt.show()

print("="*45)
print("  CELLULE 10/12 — Graphique")
print("="*45)
print("  ✅ mlflow_comparison.png sauvegardé")
print("  ✅ Cellule 10 OK — passe à la cellule 11")
print("="*45)

  CELLULE 10/12 — Graphique
  ✅ mlflow_comparison.png sauvegardé
  ✅ Cellule 10 OK — passe à la cellule 11


In [17]:
# ============================================================
# CELLULE 11/12 — Sauvegarde meilleur modèle
# Ce fichier sera utilisé par FastAPI (J10)
# ============================================================

os.makedirs("models", exist_ok=True)

# Sauvegarde locale
joblib.dump(best_model, "models/best_model.pkl")
joblib.dump(scaler,     "models/scaler.pkl")

# Log dans MLflow comme artifact final
with mlflow.start_run(run_name=f"BEST_MODEL_{best_run}"):
    mlflow.log_param("best_run",  best_run)
    mlflow.log_param("best_f1",   best_f1)
    mlflow.log_param("best_auc",  best_auc)
    mlflow.log_artifact("models/best_model.pkl")
    mlflow.log_artifact("mlflow_comparison.png")
    mlflow.sklearn.log_model(best_model, "best_model_final")
    final_run_id = mlflow.active_run().info.run_id

print("="*45)
print("  CELLULE 11/12 — Sauvegarde")
print("="*45)
print(f"  Meilleur modèle  : {best_run}")
print(f"  F1               : {best_f1}")
print(f"  AUC              : {best_auc}")
print(f"  Fichiers créés   :")
print(f"    → models/best_model.pkl ✅")
print(f"    → models/scaler.pkl     ✅")
print(f"    → mlflow_comparison.png ✅")
print(f"  Final Run ID     : {final_run_id[:8]}...")
print("="*45)
print("  ✅ Cellule 11 OK — passe à la cellule 12")
print("="*45)

2026/05/13 07:09:39 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/13 07:09:39 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


  CELLULE 11/12 — Sauvegarde
  Meilleur modèle  : RF_Baseline
  F1               : 0.864
  AUC              : 0.9732
  Fichiers créés   :
    → models/best_model.pkl ✅
    → models/scaler.pkl     ✅
    → mlflow_comparison.png ✅
  Final Run ID     : acc3dea3...
  ✅ Cellule 11 OK — passe à la cellule 12


In [18]:
# ============================================================
# CELLULE 12/12 — Résumé J12 complet
# ============================================================

print("""
╔══════════════════════════════════════════════════════════════╗
║           J12 — MLflow Tracking — TERMINÉ ✅                 ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  3 expériences loggées :                                     ║
║    ✅ RF_Baseline  → modèle de référence J11                 ║
║    ✅ RF_Optimise  → max_depth=15, n_estimators=500          ║
║    ✅ RF_Agressif  → class_weight 1:50, seuil 0.35           ║
║                                                              ║
║  Fichiers produits :                                         ║
║    ✅ models/best_model.pkl                                  ║
║    ✅ models/scaler.pkl                                      ║
║    ✅ mlflow_comparison.png                                  ║
║    ✅ mlruns/ (base MLflow locale)                           ║
║                                                              ║
╠══════════════════════════════════════════════════════════════╣
║  POUR VOIR L'INTERFACE MLFLOW :                              ║
║                                                              ║
║  1. Ouvre un nouveau terminal                                ║
║  2. cd "C:\\Users\\DELL£\\Downloads\\projet ocp"            ║
║  3. mlflow ui                                                ║
║  4. Navigateur → http://127.0.0.1:5000                       ║
║                                                              ║
║  Tu verras tes 3 expériences comparées visuellement !        ║
╚══════════════════════════════════════════════════════════════╝
""")


╔══════════════════════════════════════════════════════════════╗
║           J12 — MLflow Tracking — TERMINÉ ✅                 ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  3 expériences loggées :                                     ║
║    ✅ RF_Baseline  → modèle de référence J11                 ║
║    ✅ RF_Optimise  → max_depth=15, n_estimators=500          ║
║    ✅ RF_Agressif  → class_weight 1:50, seuil 0.35           ║
║                                                              ║
║  Fichiers produits :                                         ║
║    ✅ models/best_model.pkl                                  ║
║    ✅ models/scaler.pkl                                      ║
║    ✅ mlflow_comparison.png                                  ║
║    ✅ mlruns/ (base MLflow locale)                           ║
║                                                              ║
╠═══════════════════════════════